# Kaggle Inference-Only Notebook

This notebook does not train. It only:
1. Loads test CSV
2. Loads one or more .keras model files
3. Averages probabilities across models
4. Saves submission.csv

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
DATA_DIR = '/kaggle/input/ai-biz-2026-spring-task-3'
MODELS_DIR = '/kaggle/input/your-fold-models-dataset'

TEST_PATH = os.path.join(DATA_DIR, 'fashion-mnist_test.csv')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'fashion-mnist_test_sample_submission.csv')

# Use all .keras files in MODELS_DIR (1 file = single model inference, many files = ensemble)
MODEL_PATHS = sorted(glob.glob(os.path.join(MODELS_DIR, '*.keras')))

print('TEST_PATH exists:', os.path.exists(TEST_PATH))
print('SAMPLE_SUB_PATH exists:', os.path.exists(SAMPLE_SUB_PATH))
print('Number of models found:', len(MODEL_PATHS))
print('Example model path:', MODEL_PATHS[0] if len(MODEL_PATHS) else 'None')

In [ ]:
assert os.path.exists(TEST_PATH), f'Missing test file: {TEST_PATH}'
assert os.path.exists(SAMPLE_SUB_PATH), f'Missing sample submission: {SAMPLE_SUB_PATH}'
assert len(MODEL_PATHS) > 0, 'No .keras model files found in MODELS_DIR'

test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

X_test = test_df.values.astype(np.float32) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1)
print('X_test:', X_test.shape)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 128
AUTO = tf.data.AUTOTUNE

def preprocess_for_efficientnet(image):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.image.grayscale_to_rgb(image)
    image = tf.keras.applications.efficientnet.preprocess_input(image * 255.0)
    return image

test_ds = tf.data.Dataset.from_tensor_slices(X_test)
test_ds = test_ds.map(preprocess_for_efficientnet, num_parallel_calls=AUTO).batch(BATCH_SIZE).prefetch(AUTO)

In [ ]:
# Compatibility loader for keras version differences
def load_model_compat(model_path):
    try:
        return tf.keras.models.load_model(model_path, compile=False)
    except Exception as e:
        print('Standard load failed for:', model_path)
        print('Error:', e)

        def _patch_from_config(layer_cls):
            original = layer_cls.from_config

            @classmethod
            def patched(cls, config):
                if isinstance(config, dict):
                    config.pop('quantization_config', None)
                return original(config)

            layer_cls.from_config = patched
            return original

        patched_layers = [
            tf.keras.layers.Dense,
            tf.keras.layers.Conv2D,
            tf.keras.layers.DepthwiseConv2D,
            tf.keras.layers.SeparableConv2D,
            tf.keras.layers.BatchNormalization,
        ]

        originals = {layer: _patch_from_config(layer) for layer in patched_layers}
        try:
            model = tf.keras.models.load_model(model_path, compile=False)
        finally:
            for layer, original in originals.items():
                layer.from_config = original

        return model

In [ ]:
probs_sum = None

for i, model_path in enumerate(MODEL_PATHS, start=1):
    print(f'Loading model {i}/{len(MODEL_PATHS)}:', model_path)
    model = load_model_compat(model_path)
    probs = model.predict(test_ds, verbose=0)

    if probs_sum is None:
        probs_sum = probs
    else:
        probs_sum += probs

probs_avg = probs_sum / len(MODEL_PATHS)
preds = np.argmax(probs_avg, axis=1)
print('Predictions shape:', preds.shape)

In [ ]:
submission = sample_sub.copy()
submission['label'] = preds
output_path = '/kaggle/working/submission.csv'
submission.to_csv(output_path, index=False)
print('Saved:', output_path)
submission.head()